### Estimating Error on the CMB Optical Depth from Cleaning with WMAP and Planck

As a final project for the class, we will take a first pass at a problem that we need to solve for the CLASS project. CLASS aims to measure the optical depth to reionization ($\tau$) to its cosmic variance limit on 75% of the sky that CLASS measures. To do this we use low frequency data from WMAP and high frequency data from Planck to clean polarized synchrotron and dust emission, respectively, from our data at 93 GHz. We need to understand to what extent using the WMAP and Planck data limits us in achieving the goal of a cosmic-variance measurement of $\tau$. 

* Even though CLASS only measures 75% of the sky, in this project we will use all-sky maps to simplify the analysis.
* For simulations, use `camb` to generate angular power spectra given cosmological parameters and `healpy` `synfast` to generate maps from these spectra. For input cosmological parameters use those introduced in the `get_spectra` function of Lecture 5.
* Use `healpy` `anafast` to estimate auto and cross power spectra. 
* For foreground models, let's use `pysm`'s `s1` model for synchrotron and its `d1` model for dust.
* The low-$\ell$ ($\ell<32$) E-mode spectrum of the CMB polarization increases proportional to $\tau^2$. When estimating a best-fit $\tau$ you will use a likelihood built from a simulated E-mode angular power spectrum (the data) for $\ell<32$ and a model for the power spectrum generated by `camb` (Lecture 8).
* For simplicity, assume a Gaussian likelihood for the $C_\ell$ with no covariance between $C_\ell$. If there is time, we'll revisit this assumption in the light of arguments like the one in problem 3 of lecture 8.
* When fitting the model for $C_\ell$, you can hold all cosmological parameters but $\tau$ fixed to their input values (second bullet above). Because adjusting $\tau$ changes $C_\ell$, and therefore the cosmic variance, you will need to adjust the variance in your likelihood as well.
* You should be able to get away with NSIDE=32 healpix maps.

## Part 1: Estimate the Cosmic-Variance-Limit for Tau using CMB-only Maps

Create full sky simulations of the CMB with no foregrounds and no noise. Estimate the uncertainty on the best-fit $\tau$ by fitting many realizations of the CMB E-mode spectrum.

Instead of doing simulations, create a dataset that has its mean as the "true" power spectrum and assign errors corresponding to cosmic variance. Then use a Markov chain monte carlo to estimate the error on $\tau$.

## Part 2: Add White Noise

Now add many realizations of simulated white noise in addition to the different CMB realizations: 10, 20, 40, and 80 $\mu\mathrm{K}\,\mathrm{arcmin}$. Estimate the error on $\tau$ for these different noise levels. Try approaches with both auto and cross spectra. (For the cross spectra consider two maps with noise levels of $\sqrt{2}$ times the noise levels of the single maps.)

Instead of doing simulations, create a dataset that has its mean as the "true" tau=0.05 power spectrum and assign errors corresponding to cosmic variance plus the white noise $N_\ell$. Then use a markov chain monte carlo to estimate the error on $\tau$.

## Part 3: Add Foregrounds

Create simulations at 23 GHz (WMAP), 93 GHz (CLASS), and 353 GHz (Planck) that have dust, synchrotron, and CMB. Vary the CMB between simulations but keep the same dust and synchrotron components. Clean the 93 GHz map using ILC methods and then estimate $\tau$. Repeating this across simulations, estimate the uncertainy on $\tau$. 

## Part 4: Add White Noise to Foregrounds

Keeping the 93 GHz map noiseless, add noise to the foregrounds. For each simulation, we will create two ILC maps with independent noise and take their cross spectrum. For the first map, clean the noiseless 93 GHz map with the 23 GHz map with 400 $\mu\mathrm{K}\,\mathrm{arcmin}$ noise and a 353 GHz map with 600 $\mu\mathrm{K}\,\mathrm{arcmin}$ noise. For the second map, clean the noiseless 93 GHz map with a 30 GHz map with 400 $\mu\mathrm{K}\,\mathrm{arcmin}$ noise and a 353 GHz map with 600 $\mu\mathrm{K}\,\mathrm{arcmin}$ noise.  For each simulated cross spectrum, find the best-fit $\tau$ and report the uncertainty on $\tau$ across simulations. 


# Power Spectrum Errors

**Caveat**: I believe the arguments below pivot on assuming the errors are Gaussian, which is a good approximation for data at multipoles where $2\ell+1$ is large enough (say $30$ or so). Otherwise we need to treat the errors with a different likelihood. However, we can use this as an approximation at any $\ell$ and then research the low-$\ell$ likelihood derivation elsewhere

# Cross-Spectrum Noise

**AI Acknowledgment**: This description of errors is a lightly edited version of information dredged from publications by Gemini with some notes by me ("Toby's Notes").

The variance of the empirical cross-power spectrum between two maps (such as two Cosmic Microwave Background maps, or a CMB map and a large-scale structure tracer) for a single multipole $\ell$ is given by the following expression:
$$\text{Var}(\hat{C}_\ell^{XY}) = \frac{1}{(2\ell+1)f_{\text{sky}}} \left[ C_\ell^{XX} C_\ell^{YY} + \left(C_\ell^{XY}\right)^2 \right]$$ 
When data is averaged within a multipole bin of width $\Delta \ell$ centered at $\ell_i$, the binned variance is expressed as: [1] 
$$\sigma^2(\hat{C}_i^{XY}) = \frac{1}{(2\ell_i+1)f_{\text{sky}}\Delta \ell} \left[ C_i^{XX} C_i^{YY} + \left(C_i^{XY}\right)^2 \right]$$ 
## 1. Define Variables

* $\hat{C}_\ell^{XY}$: The estimated cross-power spectrum between map $X$ and map $Y$.
* $C_\ell^{XY}$: The true underlying theoretical cross-power spectrum between the two maps.
* $C_\ell^{XX}, C_\ell^{YY}$: The total auto-power spectra of map $X$ and map $Y$ respectively.
* $f_{\text{sky}}$: The fraction of the sky observed ($f_{\text{sky}} = 1$ for a full-sky map).
* $2\ell+1$: The total number of independent $m$-modes available at a given multipole $\ell$, representing the statistical degrees of freedom.
* $\Delta \ell$: The width of the multipole bin. [1, 2]

### 1.1 Toby's Notes

* One sometimes uses $\hat{C}_\ell$ to mean the signal in the map, but here it is used to mean both signal and noise. See below where $S_\ell$ is used for signal. The use of $N_\ell$ aligns with all convention's. 
* In the equation for $\text{Var}(\hat{C}_\ell^{XY})$, the term $C_\ell^{XX} C_\ell^{YY}$ would be present even if the two maps were uncorrelated. The $\left(C_\ell^{XY}\right)^2$ is the extra variance if the two maps have a correlation. As we will see below, the signal $S_\ell$ causes a correlation, but there can also be extra variance from correlated noise.
* That the variance is proportional to $1/f_{\text{sky}}$ makes sense qualitatively if you think of the partial sky as missing a fraction of the data that does not go into some average. For instance if the full sky had $N$ regions and the variance per some quantity in each region was $\sigma^2$, then the variance on the average would be $\sigma^2_{\text{full}}=\sigma^2/N$. If you only had a fraction of the sky, say $N-M$ where  regions, then variance on the average would be $\sigma^2_{\text{partial}}=\sigma^2/(N-M)$. Therefore
$$\sigma^2_{\text{partial}}=\left(\frac{N}{N-M}\right)\sigma^2_{\text{full}}=\frac{1}{f}\sigma^2_{\text{full}}.$$
* The variance reduction factor of $1/(2\ell+1)$ is the regular reduction in variance due to averaging over the $2\ell+1$ $a_{\ell,m}$ per $\ell$. The reduction factor $1/\Delta \ell$ factor for binned spectra similarly comes from averaging over all the $\ell$ in a bin.

## 2. Include Noise and Systematics
The auto-power terms $C_\ell^{XX}$ and $C_\ell^{YY}$ represent the total observed power, meaning they must explicitly include both the cosmological signal and any instrumental noise or residual contaminants present in each map: [2] 
$$C_\ell^{XX} = S_\ell^{XX} + N_\ell^{XX}$$ 
$$C_\ell^{YY} = S_\ell^{YY} + N_\ell^{YY}$$ 

* $S_\ell$: The true cosmological signal power spectrum.
* $N_\ell$: The instrumental noise power spectrum. [2] 

Substituting these expansion components: [2] 
$$\text{Var}(\hat{C}_\ell^{XY}) = \frac{1}{(2\ell+1)f_{\text{sky}}} \left[ (S_\ell^{XX} + N_\ell^{XX})(S_\ell^{YY} + N_\ell^{YY}) + \left(S_\ell^{XY} + N_\ell^{XY}\right)^2 \right]$$ 
If the instrumental noise realisations between map $X$ and map $Y$ are perfectly uncorrelated, the noise cross-term drops to zero ($N_\ell^{XY} = 0$), simplifying the second term to just the squared signal component. Correlated noise will also bias the expectation value of $\hat{C}_\ell^{XY}$ high. [2] 
## 3. Special Case: Map Splits with Uncorrelated Noise
When cross-correlating two different time-splits or data-splits of the same underlying CMB sky map ($X=1, Y=2$), the true signal is identical ($S_\ell^{11} = S_\ell^{22} = S_\ell^{12} = S_\ell$), while the instrumental noises are independent. The formula simplifies to: [2, 3] 
$$\text{Var}(\hat{C}_\ell^{12}) = \frac{1}{(2\ell+1)f_{\text{sky}}} \left[ S_\ell^2 + (S_\ell + N_\ell^{11})(S_\ell + N_\ell^{22}) \right]$$ 

[1] [https://link.aps.org](https://link.aps.org/doi/10.1103/PhysRevD.107.123532)
[2] [https://www.cpt.univ-mrs.fr](https://www.cpt.univ-mrs.fr/~cosmo/EC2017/Presentations/Hivon-1.pdf)
[3] [https://arxiv.org](https://arxiv.org/html/2506.20667v1)

# Case of Auto-Spectrum

**AI Acknowledgment**: This description of errors is a checked-over but essentially unaltered version of information dredged from publications by Gemini.

The variance of the empirical auto-power spectrum for a single map $X$ at a specific multipole $\ell$ is derived by setting $Y = X$ in the cross-spectrum formula:
$$\text{Var}(\hat{C}_\ell^{XX}) = \frac{2}{2\ell+1} \left(C_\ell^{XX}\right)^2$$ 
For data averaged within a multipole bin of width $\Delta \ell$ centered at $\ell_i$, the binned variance is expressed as:
$$\sigma^2(\hat{C}_i^{XX}) = \frac{2}{(2\ell_i+1)f_{\text{sky}}\Delta \ell} \left(C_i^{XX}\right)^2$$ 
## 1. Key Characteristics

* Cosmic Variance: The factor of 2 in the numerator arises because the $m$-modes are complex, and the auto-spectrum squares the map's own fluctuations. This represents the fundamental statistical limit of measuring our unique sky.
* Total Power Dependence: The term $C_\ell^{XX}$ is the total observed power, which explicitly includes both the cosmological signal ($S_\ell^{XX}$) and the instrumental noise ($N_\ell^{XX}$):

$$\text{Var}(\hat{C}_\ell^{XX}) = \frac{2}{(2\ell+1)f_{\text{sky}}} \left(S_\ell^{XX} + N_\ell^{XX}\right)^2$$ 
## 2. Expanded Component View
Expanding the squared term highlights how instrumental noise couples with the signal to increase the overall variance:
$$\text{Var}(\hat{C}_\ell^{XX}) = \frac{2}{(2\ell+1)f_{\text{sky}}} \left[ \left(S_\ell^{XX}\right)^2 + 2S_\ell^{XX}N_\ell^{XX} + \left(N_\ell^{XX}\right)^2 \right]$$ 

* $\left(S_\ell^{XX}\right)^2$: Pure cosmic variance.
* $2S_\ell^{XX}N_\ell^{XX}$: Signal-noise cross-term (sample variance of the noise).
* $\left(N_\ell^{XX}\right)^2$: Pure noise variance (dominates at high multipoles where the signal falls below the noise floor).

